# Обработка крупных наборов событий с DuckDB и Parquet

**Версия notebook для слушателя.**

Рабочая цепочка: проверка окружения → профиль качества → очистка → CSV в Parquet → SQL-агрегации → партиционирование → аналитическая витрина → выводы.

Основной набор: `data/raw/events_lite.csv`. Для расширенного режима создайте `events_standard.csv` через `scripts/generate_events.py`.

## Правила выполнения

1. Откройте весь проект, а не отдельный notebook.
2. Выполняйте ячейки сверху вниз.
3. Не загружайте весь CSV в pandas без необходимости.
4. После завершения перезапустите ядро и выполните все ячейки повторно.

In [ ]:
# При необходимости установите DuckDB в терминале:
# python -m pip install duckdb==1.5.5 pandas matplotlib jupyterlab
#
# В Google Colab раскомментируйте следующую строку:
# %pip install -q duckdb==1.5.5

## 1. Проверка окружения

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

print('Python:', sys.version.split()[0])
print('DuckDB:', duckdb.__version__)
print('pandas:', pd.__version__)
print('Рабочая папка:', Path.cwd())

In [ ]:
current_dir = Path.cwd()
if (current_dir / 'data').exists():
    project_root = current_dir
elif (current_dir.parent / 'data').exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError('Не найдена папка data. Откройте notebook внутри папки проекта.')

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
output_dir = project_root / 'outputs'
materials_dir = project_root / 'materials'
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

standard_path = raw_dir / 'events_standard.csv'
lite_path = raw_dir / 'events_lite.csv'
source_path = standard_path if standard_path.exists() else lite_path
if not source_path.exists():
    raise FileNotFoundError('Не найден events_standard.csv или events_lite.csv в data/raw')

print('Корень проекта:', project_root)
print('Источник:', source_path.name)
print('Размер CSV, МБ:', round(source_path.stat().st_size / 1024**2, 2))

In [ ]:
def sql_literal_path(path: Path) -> str:
    return path.as_posix().replace("'", "''")

source_sql = sql_literal_path(source_path)
parquet_path = processed_dir / 'events.parquet'
parquet_sql = sql_literal_path(parquet_path)
con = duckdb.connect()

csv_scan = f"read_csv('{source_sql}', header=true, auto_detect=true, nullstr='')"
print('Подключение DuckDB создано')

## 2. Первичная проверка данных

In [ ]:
row_count = con.execute(f'SELECT COUNT(*) FROM {csv_scan}').fetchone()[0]
print('Количество строк:', f'{row_count:,}')

In [ ]:
schema = con.execute(f'DESCRIBE SELECT * FROM {csv_scan}').df()
display(schema)

required_columns = {
    'event_id', 'event_time', 'event_date', 'user_id', 'session_id',
    'event_type', 'product_id', 'category', 'region', 'device',
    'quantity', 'price', 'response_ms'
}
missing_columns = required_columns - set(schema['column_name'])
assert not missing_columns, f'Не найдены поля: {sorted(missing_columns)}'

In [ ]:
sample = con.execute(f'SELECT * FROM {csv_scan} LIMIT 10').df()
display(sample)

## 3. Профиль качества

In [ ]:
quality_profile = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT event_id) AS distinct_event_ids,
    COUNT(DISTINCT user_id) AS distinct_users,
    COUNT(DISTINCT session_id) AS distinct_sessions,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END) AS missing_event_id,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS missing_region,
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS negative_price,
    SUM(CASE WHEN quantity < 0 THEN 1 ELSE 0 END) AS negative_quantity,
    SUM(CASE WHEN response_ms < 0 THEN 1 ELSE 0 END) AS negative_response_ms
FROM {csv_scan}
""").df()
display(quality_profile)

In [ ]:
duplicate_events = con.execute(f"""
SELECT event_id, COUNT(*) AS copies
FROM {csv_scan}
WHERE event_id IS NOT NULL
GROUP BY event_id
HAVING COUNT(*) > 1
ORDER BY copies DESC, event_id
LIMIT 20
""").df()
display(duplicate_events)

duplicate_group_count = con.execute(f"""
SELECT COUNT(*)
FROM (
    SELECT event_id
    FROM {csv_scan}
    WHERE event_id IS NOT NULL
    GROUP BY event_id
    HAVING COUNT(*) > 1
)
""").fetchone()[0]
print('Групп повторяющихся event_id:', duplicate_group_count)

In [ ]:
event_types_raw = con.execute(f"""
SELECT event_type, COUNT(*) AS events_count
FROM {csv_scan}
GROUP BY event_type
ORDER BY events_count DESC
""").df()
display(event_types_raw)

## 4. Очистка и создание Parquet

In [ ]:
clean_query = f"""
WITH ranked_events AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY event_id
            ORDER BY event_time
        ) AS duplicate_rank
    FROM {csv_scan}
    WHERE event_id IS NOT NULL
)
SELECT
    event_id,
    event_time,
    event_date,
    user_id,
    session_id,
    event_type,
    product_id,
    category,
    COALESCE(NULLIF(TRIM(region), ''), 'Не определён') AS region,
    device,
    quantity,
    price,
    response_ms
FROM ranked_events
WHERE duplicate_rank = 1
  AND (quantity IS NULL OR quantity >= 0)
  AND (price IS NULL OR price >= 0)
  AND (response_ms IS NULL OR response_ms >= 0)
"""

clean_row_count = con.execute(f'SELECT COUNT(*) FROM ({clean_query})').fetchone()[0]
print('Строк до очистки:', row_count)
print('Строк после очистки:', clean_row_count)
print('Исключено:', row_count - clean_row_count)

In [ ]:
if parquet_path.exists():
    parquet_path.unlink()

con.execute(f"""
COPY ({clean_query})
TO '{parquet_sql}'
(FORMAT parquet, COMPRESSION zstd)
""")

parquet_row_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{parquet_sql}')"
).fetchone()[0]
assert parquet_row_count == clean_row_count
print('Создан:', parquet_path)
print('Строк в Parquet:', parquet_row_count)

In [ ]:
size_comparison = pd.DataFrame({
    'format': ['CSV', 'Parquet'],
    'size_mb': [
        source_path.stat().st_size / 1024**2,
        parquet_path.stat().st_size / 1024**2,
    ]
}).round(2)
display(size_comparison)

## 5. Основные аналитические показатели

In [ ]:
events_by_type = con.execute(f"""
SELECT
    event_type,
    COUNT(*) AS events_count,
    COUNT(DISTINCT user_id) AS users_count,
    COUNT(DISTINCT session_id) AS sessions_count
FROM read_parquet('{parquet_sql}')
GROUP BY event_type
ORDER BY events_count DESC
""").df()
display(events_by_type)
assert events_by_type['events_count'].sum() == parquet_row_count
events_by_type.to_csv(output_dir / 'events_by_type.csv', index=False)

In [ ]:
purchases_by_region = con.execute(f"""
SELECT
    region,
    COUNT(*) AS purchase_events,
    COUNT(DISTINCT user_id) AS buyers_count,
    SUM(price * quantity) AS gross_value,
    AVG(price * quantity) AS average_purchase_event_value
FROM read_parquet('{parquet_sql}')
WHERE event_type = 'purchase'
GROUP BY region
ORDER BY gross_value DESC
""").df()
display(purchases_by_region)
purchases_by_region.to_csv(output_dir / 'purchases_by_region.csv', index=False)

In [ ]:
conversion = con.execute(f"""
WITH user_activity AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS has_view,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM read_parquet('{parquet_sql}')
    WHERE user_id IS NOT NULL
    GROUP BY user_id
)
SELECT
    SUM(has_view) AS users_with_view,
    SUM(has_purchase) AS users_with_purchase,
    SUM(CASE WHEN has_view = 1 AND has_purchase = 1 THEN 1 ELSE 0 END) AS viewers_with_purchase,
    1.0 * SUM(CASE WHEN has_view = 1 AND has_purchase = 1 THEN 1 ELSE 0 END)
        / NULLIF(SUM(has_view), 0) AS conversion_rate
FROM user_activity
""").df()
display(conversion)
assert 0 <= conversion.loc[0, 'conversion_rate'] <= 1

In [ ]:
daily_metrics = con.execute(f"""
SELECT
    event_date,
    COUNT(*) AS events_count,
    COUNT(DISTINCT user_id) AS users_count,
    COUNT(DISTINCT session_id) AS sessions_count,
    AVG(response_ms) AS average_response_ms,
    SUM(CASE WHEN event_type = 'purchase' THEN price * quantity ELSE 0 END) AS gross_value
FROM read_parquet('{parquet_sql}')
GROUP BY event_date
ORDER BY event_date
""").df()
display(daily_metrics.head(10))
assert daily_metrics['events_count'].sum() == parquet_row_count
daily_metrics.to_csv(output_dir / 'daily_metrics.csv', index=False)

## 6. Визуализации по агрегированным данным

In [ ]:
ax = daily_metrics.plot(
    x='event_date', y='events_count', figsize=(11, 5),
    legend=False, title='Количество событий по дням'
)
ax.set_xlabel('Дата')
ax.set_ylabel('Количество событий')
plt.tight_layout()
plt.show()

In [ ]:
region_chart_data = (
    purchases_by_region
    .sort_values('gross_value', ascending=False)
    .head(10)
    .sort_values('gross_value')
)
ax = region_chart_data.plot(
    x='region', y='gross_value', kind='barh', figsize=(10, 6),
    legend=False, title='Сумма покупок по регионам'
)
ax.set_xlabel('Сумма покупок')
ax.set_ylabel('Регион')
plt.tight_layout()
plt.show()

## 7. Партиционирование по дате

In [ ]:
partitioned_dir = processed_dir / 'events_by_date'
if partitioned_dir.exists():
    shutil.rmtree(partitioned_dir)
partitioned_sql = sql_literal_path(partitioned_dir)

con.execute(f"""
COPY (
    SELECT * FROM read_parquet('{parquet_sql}')
)
TO '{partitioned_sql}'
(FORMAT parquet, COMPRESSION zstd, PARTITION_BY (event_date))
""")

partition_files = sorted(partitioned_dir.rglob('*.parquet'))
print('Количество файлов:', len(partition_files))
for path in partition_files[:10]:
    print(path.relative_to(project_root))

In [ ]:
partition_glob = partitioned_dir.as_posix() + '/**/*.parquet'
partition_glob_sql = partition_glob.replace("'", "''")
selected_date = daily_metrics['event_date'].iloc[0]

date_summary = con.execute(f"""
SELECT
    event_date,
    region,
    COUNT(*) AS events_count,
    COUNT(DISTINCT user_id) AS users_count
FROM read_parquet('{partition_glob_sql}', hive_partitioning=true)
WHERE event_date = ?
GROUP BY event_date, region
ORDER BY events_count DESC
""", [selected_date]).df()
display(date_summary)
assert not date_summary.empty

In [ ]:
plan_rows = con.execute(f"""
EXPLAIN ANALYZE
SELECT region, COUNT(*) AS events_count
FROM read_parquet('{partition_glob_sql}', hive_partitioning=true)
WHERE event_date = ?
GROUP BY region
""", [selected_date]).fetchall()
for row in plan_rows:
    print(row[-1])

## 8. Аналитическая витрина

In [ ]:
mart_path = output_dir / 'daily_region_metrics.parquet'
mart_sql = sql_literal_path(mart_path)
if mart_path.exists():
    mart_path.unlink()

con.execute(f"""
COPY (
    SELECT
        event_date,
        region,
        COUNT(*) AS events_count,
        COUNT(DISTINCT user_id) AS users_count,
        COUNT(DISTINCT session_id) AS sessions_count,
        AVG(response_ms) AS average_response_ms,
        SUM(CASE WHEN event_type = 'purchase' THEN price * quantity ELSE 0 END) AS gross_value
    FROM read_parquet('{parquet_sql}')
    GROUP BY event_date, region
    ORDER BY event_date, region
)
TO '{mart_sql}'
(FORMAT parquet, COMPRESSION zstd)
""")

mart_check = con.execute(f"""
SELECT
    COUNT(*) AS rows_count,
    COUNT(DISTINCT event_date) AS dates_count,
    COUNT(DISTINCT region) AS regions_count
FROM read_parquet('{mart_sql}')
""").df()
display(mart_check)

## 9. Самостоятельный анализ региона

In [ ]:
available_regions = con.execute(f"""
SELECT DISTINCT region
FROM read_parquet('{parquet_sql}')
ORDER BY region
""").df()
display(available_regions)

selected_region = 'Москва'  # замените при необходимости

In [ ]:
region_analysis = con.execute(f"""
SELECT
    event_date,
    event_type,
    COUNT(*) AS events_count,
    COUNT(DISTINCT user_id) AS users_count,
    COUNT(DISTINCT session_id) AS sessions_count,
    AVG(response_ms) AS average_response_ms,
    SUM(CASE WHEN event_type = 'purchase' THEN price * quantity ELSE 0 END) AS gross_value
FROM read_parquet('{parquet_sql}')
WHERE region = ?
GROUP BY event_date, event_type
ORDER BY event_date, event_type
""", [selected_region]).df()

if region_analysis.empty:
    raise ValueError('По выбранному региону нет данных')
display(region_analysis.head(20))
region_analysis.to_csv(output_dir / 'region_analysis.csv', index=False)

In [ ]:
region_daily = (
    region_analysis.groupby('event_date', as_index=False)
    .agg(events_count=('events_count', 'sum'))
)
ax = region_daily.plot(
    x='event_date', y='events_count', figsize=(11, 5),
    legend=False, title=f'Активность по дням: {selected_region}'
)
ax.set_xlabel('Дата')
ax.set_ylabel('Количество событий')
plt.tight_layout()
plt.show()

## 10. Итоговые выводы слушателя

Заполните блок своими словами:

**Анализируемый файл:**  
**Количество исходных строк:**  
**Количество строк после очистки:**  
**Период:**  
**Выбранный регион:**  

**Вывод 1 и числовое подтверждение:**  

**Вывод 2 и числовое подтверждение:**  

**Обнаруженная проблема качества и способ обработки:**  

**Ограничение анализа:**  

**Рекомендуемый следующий шаг:**

## 11. Финальная проверка файлов

In [ ]:
expected_outputs = [
    output_dir / 'events_by_type.csv',
    output_dir / 'purchases_by_region.csv',
    output_dir / 'daily_metrics.csv',
    output_dir / 'daily_region_metrics.parquet',
    output_dir / 'region_analysis.csv',
]
for path in expected_outputs:
    print(('OK' if path.exists() else 'НЕ НАЙДЕН') + ':', path.name)

## Чек-лист завершения

- [ ] Notebook выполнен сверху вниз без ошибок.
- [ ] Проверены схема, пропуски, дубликаты и бизнес-правила.
- [ ] Создан `events.parquet`.
- [ ] Рассчитаны события, покупки, конверсия и дневные показатели.
- [ ] Построены две визуализации.
- [ ] Создан партиционированный набор.
- [ ] Сохранена аналитическая витрина.
- [ ] Выполнен анализ выбранного региона.
- [ ] Сформулированы два вывода и ограничение.